# ML-08 — Capstone Modeling Lane

## 1. Method choice and why

### Problem Definition

The goal is to predict whether a content page is declining in search performance.

The target variable is:

`is_declining_label`

where:

- 1 = declining content page
- 0 = non-declining content page

This is a binary classification problem.

### Model Selection

Two models are used:

1. Logistic Regression
2. Random Forest Classifier

Logistic Regression is selected as an interpretable baseline model because it provides a simple reference point.

Random Forest is selected as the main model because SEO performance signals may have non-linear relationships, and tree-based models can capture these patterns.

Model complexity is only added if it improves performance compared with the baseline.Z

In [6]:
!git clone https://github.com/Dilip-Git18/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 141 (delta 53), reused 105 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.86 MiB | 6.85 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [8]:
import os

for root, dirs, files in os.walk("flyrank-ml-internship"):
    for file in files:
        if "content_refresh" in file:
            print(os.path.join(root, file))

flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv(
    "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [12]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.columns.tolist()

Rows: 30000
Columns: 44


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [13]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


In [14]:
target_distribution = (
    df["is_declining_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(target_distribution)

df["is_declining_label"].value_counts()

is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


,count
is_declining_label,
1,16262
0,13738


In [15]:
# Remove leakage columns
# trend_direction and trend_pct are not used because they create target leakage

drop_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

X = df.drop(
    columns=drop_columns + ["is_declining_label"]
)

y = df["is_declining_label"]


# Add missing value indicators
# Helps the model learn missingness patterns safely

numeric_columns = X.select_dtypes(
    include="number"
).columns


for col in numeric_columns:
    X[col + "_missing"] = (
        X[col].isna()
    ).astype(int)


# Check feature types

numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()


categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()


print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


# Train-test split
# Stratified split keeps the same declining/non-declining ratio

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

Features shape: (30000, 69)
Target shape: (30000,)

Numeric features: 58
Categorical features: 11

Training data: (24000, 69)
Testing data: (6000, 69)


In [16]:
# Separate numeric and categorical columns after feature preparation

numeric_features = X.select_dtypes(
    include="number"
).columns


categorical_features = X.select_dtypes(
    include="object"
).columns


# Preprocessing:
# - Numeric columns: fill missing values with median
# - Categorical columns: fill missing values with most common value and encode

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    )
                ]
            ),
            numeric_features
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore"
                        )
                    )
                ]
            ),
            categorical_features
        )
    ]
)


print("Preprocessing pipeline created")

Preprocessing pipeline created


In [20]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                solver="liblinear",
                random_state=42
            )
        )
    ]
)


logistic_model.fit(
    X_train,
    y_train
)


logistic_pred = logistic_model.predict(
    X_test
)

logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]


print("Logistic Regression training completed")

Logistic Regression training completed


In [21]:
# Random Forest
# Captures non-linear relationships between SEO features

rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)


# Train model

rf_model.fit(
    X_train,
    y_train
)


# Predictions

rf_pred = rf_model.predict(
    X_test
)

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]


print("Random Forest training completed")

Random Forest training completed


In [22]:
def evaluate_model(name, y_true, prediction, probability):

    return {
        "Model": name,
        "Accuracy": round(
            accuracy_score(y_true, prediction), 4
        ),
        "Precision": round(
            precision_score(y_true, prediction), 4
        ),
        "Recall": round(
            recall_score(y_true, prediction), 4
        ),
        "F1 Score": round(
            f1_score(y_true, prediction), 4
        ),
        "ROC-AUC": round(
            roc_auc_score(y_true, probability), 4
        )
    }


# Model results

logistic_result = evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_pred,
    logistic_prob
)


rf_result = evaluate_model(
    "Random Forest",
    y_test,
    rf_pred,
    rf_prob
)


# Week-4 baseline placeholder
# Replace these values with your ML-07 baseline results

baseline_result = {
    "Model": "Week-4 Baseline",
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1 Score": None,
    "ROC-AUC": None
}


comparison_table = pd.DataFrame(
    [
        baseline_result,
        logistic_result,
        rf_result
    ]
)


comparison_table

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Week-4 Baseline,NaN,NaN,NaN,NaN,NaN
1,Logistic Regression,0.9865,0.9991,0.9760,0.9874,0.9999
2,Random Forest,0.8490,0.8342,0.9004,0.8660,0.9353


## 2. Split Design

A stratified train-test split was used because the dataset contains one row per content page.

The split keeps the same proportion of declining and non-declining pages in both training and testing data.

An 80/20 split was selected with random_state=42 for reproducibility.

A client-grouped split was not used because the prediction task focuses on content-page level classification using the available starter dataset. However, client_id was excluded from model features to prevent the model from learning client-specific patterns.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [23]:
# Analyze where Random Forest makes mistakes

errors = X_test.copy()

errors["actual"] = y_test.values
errors["prediction"] = rf_pred


# Select only incorrect predictions

wrong_predictions = errors[
    errors["actual"] != errors["prediction"]
]


print(
    "Number of incorrect predictions:",
    len(wrong_predictions)
)


# Display first 5 wrong predictions

wrong_predictions.head()

Number of incorrect predictions: 906


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,...,content_age_days_missing,age_tier_order_missing,days_since_last_update_missing,ctr_missing,avg_position_missing,engagement_rate_missing,scroll_rate_missing,ai_traffic_pct_missing,actual,prediction
2110,0.0,0.00,LOW,0.0,keyword article,informational,6466.0,42912.0,NaN,gemini-3-flash-preview,...,0,0,0,0,0,0,0,0,0,1
27201,20.0,0.13,LOW,0.0,keyword article,commercial,4162.0,24550.0,NaN,gpt-4o-mini,...,0,0,0,0,0,0,0,0,1,0
12628,0.0,0.00,LOW,0.0,keyword article,informational,4583.0,29476.0,NaN,gemini-3-flash-preview,...,0,0,0,0,0,0,0,0,1,0
17195,0.0,0.00,LOW,0.0,keyword article,informational,5572.0,37798.0,google,gemini-2.5-flash,...,0,0,0,0,0,0,0,0,0,1
17941,10.0,0.39,MEDIUM,0.0,keyword article,transactional,2363.0,16377.0,google,gemini-3-flash-preview,...,0,0,0,0,0,0,0,0,0,1


In [24]:
# Extract trained Random Forest model

rf = rf_model.named_steps["model"]


# Get feature names after preprocessing

feature_names = (
    rf_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


# Create importance dataframe

feature_importance = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": rf.feature_importances_
    }
)


# Sort by importance

feature_importance = (
    feature_importance
    .sort_values(
        by="importance",
        ascending=False
    )
)


# Show top 10 features

feature_importance.head(10)

,feature,importance
18,numeric__impressions_prev_30d,0.152854
15,numeric__impressions_last_30d,0.122692
5,numeric__impressions_90d,0.064063
25,numeric__avg_position,0.054956
13,numeric__days_with_impressions,0.048262
21,numeric__content_age_days,0.037354
4,numeric__char_count,0.027133
3,numeric__word_count,0.026900
17,numeric__sessions_last_30d,0.026562
24,numeric__ctr,0.024970


## Error Interpretation

The Random Forest model produced 906 incorrect predictions on the 6,000 test samples.

These errors represent content pages where the observed SEO signals were not enough for the model to correctly classify whether the page was declining.

Some false positives occurred where the model predicted decline, but the page was actually stable. These pages may have had weaker engagement or visibility signals that looked similar to declining pages.

Some false negatives occurred where the model predicted non-decline, but the page was actually declining. These cases may contain hidden factors not available in the dataset, such as search algorithm changes, competitor actions, or external traffic changes.

The model errors show that content performance decline is influenced by multiple signals, and prediction should be treated as decision-support rather than a perfect classification system.

## Self-check


- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.